In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
from scipy.stats.mstats import winsorize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
pd.options.display.max_columns = None


In [3]:
df = pd.read_csv('zameen-updated.csv')  # Make sure the CSV is in the same folder
df.head()

,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent,Area Type,Area Size,Area Category
0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,4 Marla,For Sale,2,02-04-2019,NaN,NaN,Marla,4.0,0-5 Marla
1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,5.6 Marla,For Sale,3,05-04-2019,NaN,NaN,Marla,5.6,5-10 Marla
2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,8 Marla,For Sale,5,07-17-2019,NaN,NaN,Marla,8.0,5-10 Marla
3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,2 Kanal,For Sale,4,04-05-2019,NaN,NaN,Kanal,2.0,1-5 Kanal
4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,For Sale,3,07-10-2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,Marla,8.0,5-10 Marla


In [4]:
print(df.shape)
df.info()
df.describe()


(168446, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168446 entries, 0 to 168445
Data columns (total 20 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   property_id    168446 non-null  int64  
 1   location_id    168446 non-null  int64  
 2   page_url       168446 non-null  object 
 3   property_type  168446 non-null  object 
 4   price          168446 non-null  int64  
 5   location       168446 non-null  object 
 6   city           168446 non-null  object 
 7   province_name  168446 non-null  object 
 8   latitude       168446 non-null  float64
 9   longitude      168446 non-null  float64
 10  baths          168446 non-null  int64  
 11  area           168446 non-null  object 
 12  purpose        168446 non-null  object 
 13  bedrooms       168446 non-null  int64  
 14  date_added     168446 non-null  object 
 15  agency         124375 non-null  object 
 16  agent          124374 non-null  object 
 17  Area Type      1

,property_id,location_id,price,latitude,longitude,baths,bedrooms,Area Size
count,1.684460e+05,168446.000000,1.684460e+05,168446.000000,168446.000000,168446.000000,168446.000000,168446.000000
mean,1.559626e+07,4375.936395,1.776576e+07,29.859519,71.239804,2.874227,3.179422,5.892188
std,2.251207e+06,3776.561581,3.531003e+07,3.807870,3.133042,2.463400,1.971401,5.778327
min,8.657500e+04,1.000000,0.000000e+00,11.052446,25.906027,0.000000,0.000000,0.000000
25%,1.488320e+07,1058.000000,1.750000e+05,24.948536,67.130363,0.000000,2.000000,3.000000
50%,1.665851e+07,3286.000000,8.500000e+06,31.459784,73.056182,3.000000,3.000000,5.000000
75%,1.708662e+07,7220.000000,1.950000e+07,33.560887,73.259870,4.000000,4.000000,8.000000
max,1.735772e+07,14220.000000,2.000000e+09,73.184088,80.161430,403.000000,68.000000,800.000000


In [5]:
df['agency'].fillna('Unknown', inplace=True)
df['agent'].fillna('Unknown', inplace=True)

df = df[df['price'] > 0]
df = df[df['baths'] >= 0]
df = df[df['bedrooms'] >= 0]
df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]

categorical_columns = ['page_url', 'property_type', 'location', 'city', 'province_name', 'purpose']
for col in categorical_columns:
    df[col] = df[col].str.strip().str.lower()

df.drop_duplicates(inplace=True)


In [6]:
df['price'] = winsorize(df['price'], limits=[0.01, 0.01])
df['latitude'] = winsorize(df['latitude'], limits=[0.05, 0.05])
df['longitude'] = winsorize(df['longitude'], limits=[0.05, 0.05])
df['baths'] = winsorize(df['baths'], limits=[0.05, 0.05])
df['bedrooms'] = winsorize(df['bedrooms'], limits=[0.05, 0.05])
df['Area Size'] = winsorize(df['Area Size'], limits=[0.05, 0.05])


In [7]:
df['price_per_sqft'] = df['price'] / df['Area Size']
df['baths_per_bedroom'] = df['baths'] / df['bedrooms'].replace(0, np.nan)
df['baths_per_bedroom'].fillna(df['baths_per_bedroom'].median(), inplace=True)

df['bedrooms_per_floor'] = df['bedrooms'] / df['Area Size']
df['bedrooms_per_floor'].fillna(df['bedrooms_per_floor'].median(), inplace=True)

df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['house_age'] = datetime.now().year - df['date_added'].dt.year
df['house_age'].fillna(df['house_age'].median(), inplace=True)


In [8]:
categorical_cols = ['property_type', 'location', 'city', 'province_name', 'purpose',
                    'agency', 'agent', 'Area Type', 'Area Category']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [9]:
features = ['Area Size', 'bedrooms', 'baths', 'house_age', 'latitude', 'longitude']
X = df[features]
y = df['price']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)


In [12]:
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name} Results:")
    print(f"MAE: {mean_absolute_error(y_true, y_pred):,.2f}")
    print(f"MSE: {mean_squared_error(y_true, y_pred):,.2f}")
    print(f"R^2: {r2_score(y_true, y_pred):.2f}")

evaluate_model("Linear Regression", y_test, y_pred_lr)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("Gradient Boosting", y_test, y_pred_gb)



Linear Regression Results:
MAE: 14,396,387.82
MSE: 565,782,319,532,547.62
R^2: 0.18

Random Forest Results:
MAE: 7,394,326.91
MSE: 292,010,298,413,947.19
R^2: 0.57

Gradient Boosting Results:
MAE: 10,156,897.09
MSE: 352,134,304,831,831.44
R^2: 0.49


In [13]:
hypothetical_data = pd.DataFrame({
    'Area Size': [3000, 1800, 2200],
    'bedrooms': [5, 3, 4],
    'baths': [4, 2, 3],
    'house_age': [12, 7, 9],
    'latitude': [32.7767, 34.0522, 35.6895],
    'longitude': [96.7970, 118.2437, 139.6917]
})

print("Future Price Predictions:")
print("Linear Regression:", lr.predict(hypothetical_data))
print("Random Forest:", rf.predict(hypothetical_data))
print("Gradient Boosting:", gb.predict(hypothetical_data))


Future Price Predictions:
Linear Regression: [-2.21531016e+09 -1.25097435e+09 -1.51198833e+09]
Random Forest: [32512050.         84826062.5        54945769.60784314]
Gradient Boosting: [26736907.61646676 19644222.95936507 29040713.83142738]
